# AST research

Working notebook for the `ast` research repo.

Submodules:

| path | upstream |
|---|---|
| `prism` | https://github.com/ruby/prism — Ruby parser producing the CRuby AST |
| `decode` | https://github.com/socketry/decode — documentation/source analysis built on top of a Ruby parser |

The kernel here is Python; Ruby work is driven through subprocesses so the submodule checkouts
stay the source of truth.

In [2]:
import os, subprocess, textwrap
from pathlib import Path

REPO = Path.home() / "usb" / "ast"
PRISM = REPO / "prism"
DECODE = REPO / "decode"

# The kernel inherits a minimal nix PATH, so put the user/system profiles back on it.
os.environ["PATH"] = os.pathsep.join([
    "/etc/profiles/per-user/nathan/bin",
    "/run/current-system/sw/bin",
    str(Path.home() / ".gem-3.4.9" / "bin"),
    os.environ.get("PATH", ""),
])

def sh(cmd, cwd=REPO):
    p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if p.returncode:
        print(f"[exit {p.returncode}] {cmd}")
    return (p.stdout + p.stderr).rstrip()

print(REPO, PRISM.is_dir(), DECODE.is_dir())
print(sh("git --version && ruby --version"))
print(sh("git submodule status"))

/home/nathan/usb/ast True True
git version 2.54.0
ruby 3.4.9 (2026-03-11 revision 76cca827ab) +PRISM [x86_64-linux]
 2e3fdd1ae90b7a866b09a1ec98ad32e9d9a71f04 decode (v0.16.0-130-g2e3fdd1)
 6cd6057eacd42ee10315568ba1bfaa181700756f prism (v1.5.2-1093-g6cd6057ea)


## Parsing with Prism

Ruby 3.4 ships Prism in the stdlib, so a quick parse needs no build step. Dumping the tree from a
Ruby subprocess keeps the notebook honest about what the real parser produces.

In [3]:
def prism_parse(source, lib=None):
    """Dump the Prism AST for `source`. lib=PRISM/'lib' uses the submodule checkout instead of stdlib."""
    load_path = f"-I{lib} " if lib else ""
    script = 'require "prism"; puts Prism.parse(ARGV[0]).value.inspect'
    return sh(f"ruby {load_path}-e {script!r} -- {source!r}")

print(prism_parse("def greet(name) = \"hi #{name}\""))

@ ProgramNode (location: (1,0)-(1,30))
├── flags: ∅
├── locals: []
└── statements:
    @ StatementsNode (location: (1,0)-(1,30))
    ├── flags: ∅
    └── body: (length: 1)
        └── @ DefNode (location: (1,0)-(1,30))
            ├── flags: newline
            ├── name: :greet
            ├── name_loc: (1,4)-(1,9) = "greet"
            ├── receiver: ∅
            ├── parameters:
            │   @ ParametersNode (location: (1,10)-(1,14))
            │   ├── flags: ∅
            │   ├── requireds: (length: 1)
            │   │   └── @ RequiredParameterNode (location: (1,10)-(1,14))
            │   │       ├── flags: ∅
            │   │       └── name: :name
            │   ├── optionals: (length: 0)
            │   ├── rest: ∅
            │   ├── posts: (length: 0)
            │   ├── keywords: (length: 0)
            │   ├── keyword_rest: ∅
            │   └── block: ∅
            ├── body:
            │   @ StatementsNode (location: (1,18)-(1,30))
            │   ├── flags: ∅
        

## Decode

`decode` is a gem, so it needs its bundle installed in the submodule before it can be driven from
here. Run the next cell once; after that `decode_rb` will work.

In [ ]:
# print(sh("bundle install", cwd=DECODE))

def decode_rb(script):
    """Run a Ruby script with decode's bundle and lib/ loaded."""
    return sh(f"bundle exec ruby -Ilib -e {script!r}", cwd=DECODE)

print(decode_rb('require "decode"; puts Decode::VERSION'))